## Undersampling for XAI validation with CLARA-style sampling + medoids

In [1]:
import numpy as np
import pandas as pd

from pyclustering.cluster.clarans import clarans
from sklearn.decomposition import PCA
from sklearn.metrics import pairwise_distances_argmin_min
from sklearn.preprocessing import StandardScaler

### Nota metodológica

Este notebook mantiene el flujo propuesto para el submuestreo representativo:

1. Estandarización separada de descriptores y embeddings.
2. Reducción por bloques con PCA.
3. Construcción de un espacio conjunto `ProtBERT + descriptores`.
4. Estratificación por clase.
5. Selección de muestras reales representativas mediante un esquema tipo `CLARA`.

Como `pyclustering` no ofrece una implementación nativa de `CLARA`, aquí se usa:

- un bucle ligero de muestreo tipo `CLARA`, y
- `pyclustering.clarans` como optimizador de medoids sobre cada muestra.

En la práctica, esto sigue siendo una selección basada en medoids y resulta mucho más utilizable en este entorno que `kmedoids` puro.

In [2]:
first_level_dir = ''
second_level_dir = ''
third_level_dir = 'G:/Informatica/IA/enzimas/datasets/test_features_sequences.csv'
y_third_dir = 'G:/Informatica/IA/enzimas/datasets/pre_processed/y_test.npy'

In [3]:
data_third = pd.read_csv(third_level_dir)
third_emb = data_third.iloc[:, :1024].to_numpy(dtype=np.float32)
third_feat = data_third.iloc[:, 1024:-1].to_numpy(dtype=np.float32)
y_third = np.load(y_third_dir)

In [4]:
def _flatten_labels(labels):
    labels = np.asarray(labels)

    if labels.ndim == 1:
        return labels

    if labels.ndim == 2:
        if labels.shape[1] == 1:
            return labels.ravel()
        return labels.argmax(axis=1)

    raise ValueError('labels debe ser un vector 1D o una matriz one-hot 2D.')


def _lookup_weight(class_weights, cls):
    if cls in class_weights:
        return class_weights[cls]

    try:
        cls_int = int(cls)
        if cls_int in class_weights:
            return class_weights[cls_int]
        cls_str = str(cls_int)
        if cls_str in class_weights:
            return class_weights[cls_str]
    except (TypeError, ValueError):
        pass

    cls_str = str(cls)
    if cls_str in class_weights:
        return class_weights[cls_str]

    raise KeyError(f'No se encontró un peso para la clase {cls!r}.')


def recommend_class_weights(labels, n_select, strategy='proportional'):
    labels = _flatten_labels(labels)
    classes, counts = np.unique(labels, return_counts=True)
    k = int(n_select)

    if k < len(classes):
        raise ValueError('n_select debe ser al menos igual al número de clases.')

    if strategy == 'proportional':
        return None

    if strategy == 'balanced':
        quotas = np.full(len(classes), k // len(classes), dtype=int)
        quotas[:k % len(classes)] += 1
        quotas = np.minimum(quotas, counts)

        missing = k - int(quotas.sum())
        while missing > 0:
            capacity = counts - quotas
            available = np.flatnonzero(capacity > 0)
            if len(available) == 0:
                break
            best = available[int(np.argmax(capacity[available]))]
            quotas[best] += 1
            missing -= 1

        return dict(zip(classes.tolist(), quotas.tolist()))

    if strategy == 'keep_all_minority':
        if len(classes) != 2:
            raise ValueError("'keep_all_minority' solo está disponible para clasificación binaria.")

        minority_pos = int(np.argmin(counts))
        majority_pos = 1 - minority_pos
        minority_count = int(counts[minority_pos])
        majority_count = int(counts[majority_pos])

        if minority_count > k:
            raise ValueError('La clase minoritaria tiene más muestras que el presupuesto total.')

        quotas = np.zeros(2, dtype=int)
        quotas[minority_pos] = minority_count
        quotas[majority_pos] = k - minority_count

        if quotas[majority_pos] > majority_count:
            raise ValueError('No hay suficientes muestras en la clase mayoritaria para completar el presupuesto.')

        return dict(zip(classes.tolist(), quotas.tolist()))

    raise ValueError(f'Estrategia no soportada: {strategy!r}')


def _normalize_class_weights(labels, class_weights=None):
    classes, counts = np.unique(labels, return_counts=True)

    if class_weights is None:
        raw = counts.astype(float)
    elif isinstance(class_weights, dict):
        raw = np.array([_lookup_weight(class_weights, cls) for cls in classes], dtype=float)
    else:
        raw = np.asarray(class_weights, dtype=float)
        if raw.shape[0] != len(classes):
            raise ValueError(
                f'class_weights debe tener longitud {len(classes)} (una por clase en np.unique(labels)).'
            )

    if np.any(raw < 0):
        raise ValueError('class_weights no puede contener valores negativos.')
    if raw.sum() == 0:
        raise ValueError('class_weights no puede sumar 0.')

    return classes, counts.astype(int), raw / raw.sum()


def _allocate_quotas(labels, k, class_weights=None):
    labels = _flatten_labels(labels)
    classes, counts, target = _normalize_class_weights(labels, class_weights=class_weights)

    if k > len(labels):
        raise ValueError('k no puede ser mayor que el número total de muestras.')
    if k < len(classes):
        raise ValueError('k debe permitir al menos una muestra por clase.')

    raw = target * k
    quotas = np.floor(raw).astype(int)
    quotas = np.maximum(quotas, 1)
    quotas = np.minimum(quotas, counts)

    diff = k - int(quotas.sum())

    while diff > 0:
        capacity = counts - quotas
        candidates = np.flatnonzero(capacity > 0)
        if len(candidates) == 0:
            raise RuntimeError('No quedan clases con capacidad para completar el cupo total.')
        priority = raw[candidates] - quotas[candidates]
        best = candidates[int(np.argmax(priority))]
        quotas[best] += 1
        diff -= 1

    while diff < 0:
        removable = np.flatnonzero(quotas > 1)
        if len(removable) == 0:
            raise RuntimeError('No es posible reducir cuotas sin dejar una clase vacía.')
        surplus = quotas[removable] - raw[removable]
        best = removable[int(np.argmax(surplus))]
        quotas[best] -= 1
        diff += 1

    if int(quotas.sum()) != k:
        raise RuntimeError(f'No se pudo repartir exactamente k={k}.')

    return classes, quotas.astype(int), counts.astype(int)


def prepare_joint_space(
    descriptors,
    embeddings,
    pca_desc_components=50,
    pca_emb_components=50,
    descriptor_weight=1.0,
    embedding_weight=1.0,
    random_state=42,
):
    descriptors = np.asarray(descriptors, dtype=np.float32)
    embeddings = np.asarray(embeddings, dtype=np.float32)

    if descriptors.shape[0] != embeddings.shape[0]:
        raise ValueError('descriptors y embeddings deben tener el mismo número de filas.')

    desc_scaled = StandardScaler().fit_transform(descriptors)
    emb_scaled = StandardScaler().fit_transform(embeddings)

    if pca_desc_components is not None and desc_scaled.shape[1] > pca_desc_components:
        n_comp = min(pca_desc_components, desc_scaled.shape[0] - 1, desc_scaled.shape[1])
        if n_comp >= 1:
            desc_scaled = PCA(n_components=n_comp, random_state=random_state).fit_transform(desc_scaled)

    if pca_emb_components is not None and emb_scaled.shape[1] > pca_emb_components:
        n_comp = min(pca_emb_components, emb_scaled.shape[0] - 1, emb_scaled.shape[1])
        if n_comp >= 1:
            emb_scaled = PCA(n_components=n_comp, random_state=random_state).fit_transform(emb_scaled)

    desc_scaled = desc_scaled / max(np.sqrt(desc_scaled.shape[1]), 1.0)
    emb_scaled = emb_scaled / max(np.sqrt(emb_scaled.shape[1]), 1.0)

    return np.concatenate(
        [descriptor_weight * desc_scaled, embedding_weight * emb_scaled],
        axis=1,
    ).astype(np.float32)


def _compute_total_cost(X, medoid_points):
    _, distances = pairwise_distances_argmin_min(X, medoid_points, metric='euclidean')
    return float(distances.sum())


def _resolve_clara_sample_size(n_samples, k, clara_sample_size):
    if clara_sample_size == 'auto':
        clara_sample_size = min(n_samples, max(k + 25, int(np.ceil(1.15 * k))))
    clara_sample_size = int(clara_sample_size)
    clara_sample_size = max(clara_sample_size, k + 2)
    return min(clara_sample_size, n_samples)


def _resolve_clarans_neighbors(k, clarans_maxneighbor):
    if clarans_maxneighbor == 'auto':
        clarans_maxneighbor = max(5, min(50, k // 2))
    return int(clarans_maxneighbor)


def _fit_medoids_with_clarans(X, k, clarans_numlocal=1, clarans_maxneighbor='auto'):
    maxneighbor = _resolve_clarans_neighbors(k, clarans_maxneighbor)
    model = clarans(
        data=X.tolist(),
        number_clusters=int(k),
        numlocal=int(clarans_numlocal),
        maxneighbor=maxneighbor,
    )
    model.process()
    medoids = np.asarray(model.get_medoids(), dtype=int)
    medoids = np.unique(medoids)

    if medoids.shape[0] != k:
        raise RuntimeError('pyclustering.clarans no devolvió exactamente k medoids únicos.')

    return medoids


def clara_medoid_indices(
    X,
    k,
    clara_sample_size='auto',
    clara_trials=3,
    clarans_numlocal=1,
    clarans_maxneighbor='auto',
    random_state=42,
):
    X = np.asarray(X, dtype=np.float32)
    n_samples = X.shape[0]

    if k >= n_samples:
        return np.arange(n_samples, dtype=int), 0.0

    sample_size = _resolve_clara_sample_size(n_samples, k, clara_sample_size)
    rng = np.random.default_rng(random_state)

    best_medoids = None
    best_cost = np.inf

    for _ in range(int(clara_trials)):
        sample_idx = rng.choice(n_samples, size=sample_size, replace=False)
        local_medoids = _fit_medoids_with_clarans(
            X[sample_idx],
            k=k,
            clarans_numlocal=clarans_numlocal,
            clarans_maxneighbor=clarans_maxneighbor,
        )
        candidate_medoids = sample_idx[local_medoids]
        candidate_cost = _compute_total_cost(X, X[candidate_medoids])

        if candidate_cost < best_cost:
            best_cost = candidate_cost
            best_medoids = candidate_medoids.copy()

    best_medoids = np.asarray(best_medoids, dtype=int)

    if np.unique(best_medoids).shape[0] != k:
        raise RuntimeError('CLARA-style selection no devolvió exactamente k medoids únicos.')

    return np.sort(best_medoids), float(best_cost)


def stratified_clara_kmedoids_subsample(
    descriptors,
    embeddings,
    labels,
    n_select,
    class_weights=None,
    pca_desc_components=50,
    pca_emb_components=50,
    descriptor_weight=1.0,
    embedding_weight=1.0,
    clara_sample_size='auto',
    clara_trials=3,
    clarans_numlocal=1,
    clarans_maxneighbor='auto',
    random_state=42,
    return_metadata=False,
):
    labels = _flatten_labels(labels)
    n_total = len(labels)

    if isinstance(n_select, float):
        if not (0 < n_select <= 1):
            raise ValueError('Si n_select es float, debe estar en (0, 1].')
        k = int(np.round(n_total * n_select))
    else:
        k = int(n_select)

    k = max(1, min(k, n_total))

    X = prepare_joint_space(
        descriptors,
        embeddings,
        pca_desc_components=pca_desc_components,
        pca_emb_components=pca_emb_components,
        descriptor_weight=descriptor_weight,
        embedding_weight=embedding_weight,
        random_state=random_state,
    )

    classes, quotas, counts = _allocate_quotas(labels, k, class_weights=class_weights)

    selected_indices = []
    class_rows = []

    for offset, (cls, quota, count) in enumerate(zip(classes, quotas, counts)):
        cls_idx = np.flatnonzero(labels == cls)

        if quota >= len(cls_idx):
            chosen = cls_idx.copy()
            class_cost = 0.0
        else:
            local_selected, class_cost = clara_medoid_indices(
                X[cls_idx],
                k=quota,
                clara_sample_size=clara_sample_size,
                clara_trials=clara_trials,
                clarans_numlocal=clarans_numlocal,
                clarans_maxneighbor=clarans_maxneighbor,
                random_state=random_state + offset,
            )
            chosen = cls_idx[local_selected]

        selected_indices.extend(chosen.tolist())
        class_rows.append(
            {
                'class': cls,
                'available': int(count),
                'selected': int(len(chosen)),
                'selection_rate': float(len(chosen) / count),
                'clara_cost': float(class_cost),
            }
        )

    selected_indices = np.asarray(sorted(selected_indices), dtype=int)

    if selected_indices.shape[0] != k:
        raise RuntimeError(
            f'La selección final debería contener {k} muestras y devolvió {selected_indices.shape[0]}.'
        )

    class_summary = pd.DataFrame(class_rows)

    if return_metadata:
        return {
            'selected_indices': selected_indices,
            'class_summary': class_summary,
            'joint_space_shape': X.shape,
            'quotas': dict(zip(classes.tolist(), quotas.tolist())),
        }

    return selected_indices

### Uso en binario y multiclase

La función `stratified_clara_kmedoids_subsample(...)` puede usarse tanto para clasificación binaria como multiclase.

- `labels` admite vector 1D o etiquetas one-hot 2D.
- `class_weights=None` preserva la distribución original.
- `recommend_class_weights(..., strategy='balanced')` ayuda cuando quieres comparar explicaciones entre clases con cobertura más uniforme.
- `recommend_class_weights(..., strategy='keep_all_minority')` está pensado para binario con desbalance extremo.

#### Recomendación para el 3er nivel

En el 3er nivel, donde la clase positiva es muy escasa, la recomendación es conservar todas las muestras minoritarias y completar el presupuesto con medoids de la clase mayoritaria. Eso evita que la muestra final de 300 casos deje a la clase rara casi ausente al comparar SHAP e Integrated Gradients.

In [5]:
# Caso recomendado para el 3er nivel: conservar la clase minoritaria completa.
#
# Otros ejemplos:
# - Proporcional: class_weights = recommend_class_weights(y, 300, strategy='proportional')
# - Balanceado binario o multiclase: class_weights = recommend_class_weights(y, 300, strategy='balanced')
# - Multiclase manual: class_weights = {0: 40, 1: 40, 2: 45, 3: 45, 4: 45, 5: 45, 6: 40}

third_level_weights = recommend_class_weights(
    y_third,
    n_select=300,
    strategy='keep_all_minority',
)

selection = stratified_clara_kmedoids_subsample(
    descriptors=third_feat,
    embeddings=third_emb,
    labels=y_third,
    n_select=300,
    class_weights=third_level_weights,
    pca_desc_components=30,
    pca_emb_components=30,
    descriptor_weight=1.0,
    embedding_weight=1.0,
    clara_sample_size='auto',
    clara_trials=2,
    clarans_numlocal=1,
    clarans_maxneighbor='auto',
    random_state=42,
    return_metadata=True,
)

indexes = selection['selected_indices']
selected_labels = _flatten_labels(y_third)[indexes]
selected_subset = data_third.iloc[indexes].copy()
selected_subset.insert(0, 'original_index', indexes)
selected_subset['label'] = selected_labels

In [6]:
print(f'Total de muestras seleccionadas: {len(indexes)}')
print('Dimensión del espacio conjunto:', selection['joint_space_shape'])
print('Cuotas utilizadas:', selection['quotas'])
print('\nResumen por clase:')
display(selection['class_summary'])

print('\nPrimeros índices seleccionados:')
print(indexes[:20])

print('\nPrimeras secuencias seleccionadas:')
display(selected_subset[['original_index', 'sequence', 'label']].head())

# Si quieres guardar el subconjunto final para XAI, descomenta esta línea.
# selected_subset.to_csv('datasets/xai_selected_subset.csv', index=False)

Total de muestras seleccionadas: 300
Dimensión del espacio conjunto: (3079, 60)
Cuotas utilizadas: {0.0: 254, 1.0: 46}

Resumen por clase:


,class,available,selected,selection_rate,clara_cost
0,0.0,3033,254,0.083745,11473.991729
1,1.0,46,46,1.000000,0.000000



Primeros índices seleccionados:
[ 14  35  37  44  55  73  88  97 101 108 111 124 130 135 147 159 169 174
 177 208]

Primeras secuencias seleccionadas:


,original_index,sequence,label
14,14,MSGSYGGGGYGGRGGGGGGYSNGYDRNGGGYSNNYSSHGGSNGYGG...,0.0
35,35,MATVPSTKLELTVSKTSDWNTGYDGQFKLENKNDYDILQWGMTFDF...,1.0
37,37,MSDDSSDSTDAPGTSRSRDSGLTERQRTILDVIRASVTTRGYPPSI...,0.0
44,44,MDLTDDSPDGEVKIHKKDFHHPYTPYPIQEKFMQTVYDVLEQGKIG...,0.0
55,55,MTNGYLFREYIGAQFTGVRFSDVPINPNLSFNFILSFAIDYTSPAG...,1.0
